# FastVLA: Ultra-Efficient VLA Fine-Tuning on Tesla T4 (Distributed)

FastVLA is designed to democratize Vision-Language-Action (VLA) models. This notebook supports **distributed training on Kaggle's 2x T4 GPUs** with automatic mixed precision and gradient accumulation.

### Why this matters?
- Standard OpenVLA-7B (FP16) requires ~28GB VRAM (impossible on T4).
- FastVLA uses 4-bit QLoRA and Custom Triton Kernels to reduce memory by 70%.
- **Distributed Training**: Automatically utilizes both T4 GPUs on Kaggle.
- You get 6x faster iterations than standard implementations.

> **Goal:** Fine-tune OpenVLA for 50 steps on the PushT robotics dataset in under 2 minutes.

## ⚠️ Important: Gated Model Access (Llama-2)

OpenVLA internally uses **Llama-2-7B** as its language backbone. Meta requires all users to manually request access to Llama-2. 

**If you haven't been approved yet:**
1. Visit [meta-llama/Llama-2-7b-hf](https://huggingface.co/meta-llama/Llama-2-7b-hf) and click **Request Access**.
2. **Wait for Approval**: This can take anywhere from 1 hour to 2 days.
3. **Login**: Once approved, use your HF token in the cell below.

**🚀 Want to skip the wait? (Instant Mode)**
If you want to start training **immediately** without waiting for Meta's approval, simply change the `model_id` in Step 2 to `"smolvla"`. It is non-gated and works instantly!

## 0. Authentication
1. **Add Token**: 
   - **Kaggle**: Go to **Add-ons -> Secrets**, add `HF_TOKEN` with your HuggingFace API key.
   - **Colab**: Click the **🔑 (Secrets)** icon, add `HF_TOKEN` and enable 'Notebook access'.

In [1]:
# ── 1. Setup & Installation ───────────────────────────────────────────────
# We pin numpy and upgrade huggingface_hub BEFORE importing anything
!pip install -q "numpy>=1.24.0,<2.2.0" "huggingface_hub>=0.30.0" --no-cache-dir
!pip install -q unsloth_zoo git+https://github.com/unslothai/unsloth.git --no-cache-dir
!pip install -q "fastvla[all] @ git+https://github.com/BouajilaHamza/FastVLA.git" --no-cache-dir
!pip install -q --upgrade bitsandbytes accelerate peft transformers datasets timm --no-cache-dir

print("✅ Installation complete. Please restart session if you encounter import errors.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 187.0 MB/s eta 0:00:00 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 112.0 MB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 372.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.4.4 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 4.8.4 which is incompatible.
unsloth-zoo 2026.4.4 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!

## 1. Setup Environment
We install FastVLA and its optimized dependencies. 

**Tip for Kaggle users:** You can attach the 'openvla' model directly from the '+ Add Model' tab in the right sidebar to bypass the 15GB download and start training instantly!

In [2]:
# ── 2. Authentication & Verification ───────────────────────────────────────
import os
import torch
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=token)
except:
    try:
        from google.colab import userdata
        login(token=userdata.get("HF_TOKEN"))
    except:
        login()

# Verify FastVLA
try:
    import unsloth
    print("✓ Unsloth patched")
except: pass

import fastvla
from fastvla import model as fv_model
from fastvla.kernels import TRITON_AVAILABLE

print(f"✓ FastVLA {fastvla.__version__} Ready")
print(f"  - Unsloth Optimizer: {'✓ Enabled' if fv_model.UNSLOTH_AVAILABLE else 'ℹ Disabled (using BnB fallback)'}")
print(f"  - Triton Kernels:    {'✓ Active' if TRITON_AVAILABLE else '✗ Inactive (using PyTorch fallback)'}")
print(f"  - Device:            {torch.cuda.get_device_name(0)}")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✓ Unsloth patched

FastVLA Health Check
  PyTorch:    2.10.0+cu128
  CUDA:       12.8
  Device:     Tesla T4
  Unsloth:    ✓ Available
  BnB:        ✓ Available
  Triton:     ✓ Available
  GPU Memory: 0.01GB allocated, 0.02GB reserved, 0.01GB peak

✓ FastVLA 0.1.6 Ready
  - Unsloth Optimizer: ✓ Enabled
  - Triton Kernels:    ✓ Active
  - Device:            Tesla T4


## 2. Load Model in 4-bit (QLoRA)
We load OpenVLA-7B with 4-bit quantization. 

**Important:** The `action_dim` parameter must match your dataset's action dimensions. For PushT, this is 2 (x, y coordinates). For other robotics datasets, it's typically 7 (x, y, z, roll, pitch, yaw, gripper).

**Instant Start Tip:** To skip Llama-2 gated access issues, change `model_id` below to `"smolvla"`.

In [3]:
from fastvla import FastVLAModel
import torch

# Our registry recognizes "openvla-7b" and "smolvla"
# You can also pass the full HF ID "openvla/openvla-7b"
model_id = "openvla-7b" 

# IMPORTANT: action_dim must match your dataset. PushT uses 2D actions (x, y).
ACTION_DIM = 2 

print(f"Loading {model_id} in 4-bit...")
model = FastVLAModel.from_pretrained(
    model_id,
    load_in_4bit=True,
    use_peft=True,
    gradient_checkpointing=True,
    action_dim=ACTION_DIM,
)

# The new model.py includes memory diagnostics and health checks
print(f"Model loaded. {model.config.llm_name} is active.")


Loading openvla-7b in 4-bit...

FastVLA Health Check
  PyTorch:    2.10.0+cu128
  CUDA:       12.8
  Device:     Tesla T4
  Unsloth:    ✓ Available
  BnB:        ✓ Available
  Triton:     ✓ Available
  GPU Memory: 0.01GB allocated, 0.02GB reserved, 0.01GB peak



[fastvla.model|WARNING]Initial vision load failed for openvla/openvla-7b: Composite VLM detected (openvla/openvla-7b). Triggering recovery.... Attempting recovery...
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/openvla/openvla-7b/resolve/main/adapter_config.json
[huggingface_hub.utils._http|WARNING]HTTP Error 503 thrown while requesting HEAD https://huggingface.co/openvla/openvla-7b/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].
[huggingface_hub.utils._http|WARNING]Retrying in 1s [Retry 1/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/openvla/openvla-7b/resolve/main/adapter_config.json
[huggingface_hub.utils._http|WARNING]HTTP Error 503 thrown while requesting HEAD https://huggingface.co/openvla/openvla-7b/resolve/main/adapter_config.json
Retrying in 2s [Retry 2/5].
[huggingface_hub.utils._http|WARNING]Retrying in 2s [Retry 2/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/openvla/openvla-7b/resolve/main

Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

HTTP Error 500 thrown while requesting HEAD https://huggingface.co/unsloth/llama-2-7b-bnb-4bit/resolve/main/adapter_config.json
[huggingface_hub.utils._http|WARNING]HTTP Error 500 thrown while requesting HEAD https://huggingface.co/unsloth/llama-2-7b-bnb-4bit/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].
[huggingface_hub.utils._http|WARNING]Retrying in 1s [Retry 1/5].


==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-2-7b-bnb-4bit as a legacy tokenizer.
[fastvla.model|WARNING]Unsloth LLM load failed: Unsloth: Rank of LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.18.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'o_proj', 'q_proj', 'gate_proj', 'up_proj', 'down_proj', 'k_proj', 'v_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, ta

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded. meta-llama/Llama-2-7b-hf is active.


## 3. Fine-Tuning on PushT (Real Robotics)
Next, we load the lerobot/pusht_image dataset and run an optimized training loop with distributed training support.

**Distributed Training Features:**
- Automatically uses both T4 GPUs on Kaggle
- Mixed precision (fp16) for faster training
- 8-bit optimizer for memory efficiency
- Gradient accumulation for larger effective batch sizes

In [4]:
from fastvla import FastVLATrainer, get_dataset

# Load only a small subset for demonstration
dataset = get_dataset("lerobot/pusht_image")

# Verify action dimensions match
sample_action = dataset[0]["actions"]
print(f"Dataset action shape: {sample_action.shape}")
print(f"Model action_dim: {model.config.action_dim}")
assert sample_action.shape[-1] == model.config.action_dim, \
    f"Action dimension mismatch! Dataset: {sample_action.shape[-1]}, Model: {model.config.action_dim}"

trainer = FastVLATrainer(
    model=model,
    dataset=dataset,
    batch_size=2,  # Per-GPU batch size
    lr=1e-4,
    max_steps=50,  # Set to 2000 for full convergence
    # Distributed training optimizations
    use_mixed_precision=True,  # fp16 for T4 GPUs
    use_8bit_optimizer=True,   # Memory efficient
    gradient_accumulation_steps=2,  # Effective batch size = 4
    # Checkpointing and logging
    output_dir="./checkpoints",
    save_steps=50,
    logging_steps=10,
)

print("Starting Distributed Training...")
results = trainer.train()

print(f"Training Complete! Final loss: {results[-1]['loss']:.4f}")

📥 Loading dataset lerobot/pusht_image from HuggingFace...


Dataset action shape: torch.Size([2])
Model action_dim: 2
Starting Distributed Training...


Epoch 0:   0%|          | 0/500 [00:01<?, ?it/s]


ValueError: You have to specify input_ids

## 4. Inference (Predict Action)
Now we test the model by giving it an image and a text prompt to predict the robot's next action tensor.

In [ ]:
import numpy as np
from PIL import Image

# Dummy image for demonstration
dummy_image = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))
prompt = "push the t-shaped block into the goal zone"

# Process the image and prompt through the model
from torchvision import transforms
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

pixel_values = transform(dummy_image).unsqueeze(0).unsqueeze(0)  # [1, 1, C, H, W]
input_ids = model.tokenizer(prompt, return_tensors="pt")["input_ids"]

with torch.no_grad():
    action, _ = model(pixel_values=pixel_values, input_ids=input_ids)

print(f"Predicted Action Tensor ({model.config.action_dim}D):")
print(action)

---
**Star the repo** to support democratized robotics!
[GitHub: FastVLA](https://github.com/BouajilaHamza/FastVLA)

**Created by the FastVLA Team.**